








































# 08 — Equal-Label-Budget: Calibrate or Adapt?

Given an identical labelled buffer from the target (1/5/10% of target train, stratified by Attack), compares five uses of the same budget under cross-dataset transfer, both directions, rf/lgbm/mlp.

Buffer ordering: NF-v2 carries no timestamps, so buffers are random stratified samples from the target train partition (disjoint from the evaluation split; the few-shot-target-labels setting). Chronological buffer ordering is covered by the original CIC 2018-to-2019 experiments; this notebook answers the label-efficiency question on the extractor-constant pair.

Strategies: zero_shot (source model, no budget); calibrate (frozen source model + Platt sigmoid fitted on the buffer); buffer_only (fresh model on buffer alone); augment (retrain on source train + buffer); weighted_augment (buffer duplicated to ~25% of training mass, K = |source|/(3|buffer|), seed 42 only); oracle (full target-train retrain, upper bound, seed 42 only). Imputation medians are fitted on each strategy's own training data; calibrate keeps source medians. Seeds 42-44 for the core strategies. Resume-append on (seed, source, target, model, budget, strategy).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, time, json
import numpy as np
import pandas as pd

BASE   = '/content/drive/MyDrive/drift-conference'
CACHE  = f'{BASE}/data/nfv2/cache'
RESULT = f'{BASE}/results/nfv2'
os.makedirs(RESULT, exist_ok=True)

CFG = dict(
    seeds         = [42, 43, 44],
    ref_seed      = 42,            # weighted_augment + oracle run at this seed only
    budgets       = [0.01, 0.05, 0.10],
    test_size     = 0.30,
    rf_estimators = 300,
    ece_bins      = 15,
    mlp_hidden    = (128, 64),
    mlp_max_iter  = 100,
)
BUDGET_CSV = f'{RESULT}/nfv2_budget_strategies.csv'
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=2))

In [ ]:
d18 = pd.read_parquet(f'{CACHE}/nf2018v2_prepared.parquet')
dun = pd.read_parquet(f'{CACHE}/nfunswv2_prepared.parquet')
DATASETS = {'nf2018': d18, 'nfunsw': dun}
DIRECTIONS = [('nf2018', 'nfunsw'), ('nfunsw', 'nf2018')]
FEATURES = [c for c in d18.columns if c not in ('Label', 'Attack')]
assert [c for c in dun.columns if c not in ('Label', 'Attack')] == FEATURES
print('features:', len(FEATURES), '| nf2018:', d18.shape, '| nfunsw:', dun.shape)

In [ ]:
from sklearn.metrics import (f1_score, matthews_corrcoef, average_precision_score,
                              brier_score_loss, confusion_matrix)

def ece_score(y_true, p_pos, n_bins=15):
    conf = np.maximum(p_pos, 1 - p_pos)
    correct = ((p_pos >= 0.5).astype(int) == y_true).astype(float)
    bins = np.linspace(0.5, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return ece

def all_metrics(y_true, p_pos):
    y_true = np.asarray(y_true); p_pos = np.asarray(p_pos)
    pred = (p_pos >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    ap_att = average_precision_score(y_true, p_pos)
    ap_ben = average_precision_score(1 - y_true, 1 - p_pos)
    return dict(
        macro_f1    = f1_score(y_true, pred, average='macro'),
        weighted_f1 = f1_score(y_true, pred, average='weighted'),
        mcc         = matthews_corrcoef(y_true, pred),
        auprc_macro = (ap_att + ap_ben) / 2,
        fp_rate     = fp / (fp + tn) if (fp + tn) else np.nan,
        brier       = brier_score_loss(y_true, p_pos),
        ece         = ece_score(y_true, p_pos, CFG['ece_bins']),
    )

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

F32_SAFE = 1e37

def clean_X(df, medians=None):
    X = df[FEATURES].astype('float64')
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.mask(X.abs() > F32_SAFE, np.nan)
    if medians is None:
        medians = X.median()
    return X.fillna(medians), medians

def make_model(name, seed):
    if name == 'rf':
        return RandomForestClassifier(n_estimators=CFG['rf_estimators'],
                                      n_jobs=-1, random_state=seed)
    if name == 'lgbm':
        return lgb.LGBMClassifier(n_estimators=CFG['rf_estimators'],
                                  random_state=seed, n_jobs=-1, verbosity=-1)
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=CFG['mlp_hidden'],
                              activation='relu', solver='adam', batch_size=1024,
                              max_iter=CFG['mlp_max_iter'], early_stopping=True,
                              validation_fraction=0.05, n_iter_no_change=5,
                              random_state=seed)),
    ])

def platt_fit(scores, y):
    lr = LogisticRegression(max_iter=1000)
    lr.fit(np.asarray(scores).reshape(-1, 1), np.asarray(y))
    return lr

def platt_apply(lr, scores):
    return lr.predict_proba(np.asarray(scores).reshape(-1, 1))[:, 1]

def stratified_buffer(train_df, frac, seed):
    return (train_df.groupby('Attack', group_keys=False)
            .apply(lambda g: g.sample(max(1, int(round(len(g) * frac))),
                                      random_state=seed))
            .reset_index(drop=True))

In [ ]:
def record(row):
    pd.DataFrame([row]).to_csv(BUDGET_CSV, mode='a', index=False,
                               header=not os.path.exists(BUDGET_CSV))

def fit_eval(mname, seed, train_X, train_y, test_X, test_y):
    model = make_model(mname, seed)
    t0 = time.time()
    model.fit(train_X, train_y)
    p = model.predict_proba(test_X)[:, 1]
    m = all_metrics(test_y, p)
    m['fit_s'] = round(time.time() - t0)
    del model
    gc.collect()
    return m

done = set()
if os.path.exists(BUDGET_CSV):
    prev = pd.read_csv(BUDGET_CSV)
    done = set(map(tuple, prev[['seed', 'source', 'target', 'model',
                                'budget', 'strategy']].astype(str).values))
    print(f'resume: {len(done)} rows already recorded')

def is_done(seed, src, tgt, mname, budget, strat):
    return (str(seed), src, tgt, mname, str(budget), strat) in done

def mark(seed, src, tgt, mname, budget, strat, metrics):
    row = dict(seed=seed, source=src, target=tgt, model=mname,
               budget=budget, strategy=strat, **metrics)
    record(row)
    done.add((str(seed), src, tgt, mname, str(budget), strat))
    print(f"  s{seed} {src}->{tgt} {mname} b={budget} {strat}: "
          f"MCC={metrics['mcc']:.3f} macroF1={metrics['macro_f1']:.3f} "
          f"ECE={metrics['ece']:.3f}")

for seed in CFG['seeds']:
    splits = {}
    for name, d in DATASETS.items():
        tr, te = train_test_split(d, test_size=CFG['test_size'],
                                  stratify=d['Attack'], random_state=seed)
        splits[name] = dict(train=tr.reset_index(drop=True),
                            test=te.reset_index(drop=True))
    for src, tgt in DIRECTIONS:
        te = splits[tgt]['test']
        yte = te['Label'].values
        for mname in ('rf', 'lgbm', 'mlp'):
            need_source = (not is_done(seed, src, tgt, mname, 0, 'zero_shot')) or \
                          any(not is_done(seed, src, tgt, mname, b, 'calibrate')
                              for b in CFG['budgets'])
            src_model, src_med, p_te_src = None, None, None
            if need_source:
                Xtr, src_med = clean_X(splits[src]['train'])
                ytr = splits[src]['train']['Label'].values
                src_model = make_model(mname, seed)
                t0 = time.time()
                src_model.fit(Xtr, ytr)
                print(f'seed {seed} | source fit {mname} on {src}: {time.time()-t0:.0f}s')
                del Xtr
                gc.collect()
                Xte_src, _ = clean_X(te, medians=src_med)
                p_te_src = src_model.predict_proba(Xte_src)[:, 1]
                del Xte_src
                gc.collect()
                if not is_done(seed, src, tgt, mname, 0, 'zero_shot'):
                    mark(seed, src, tgt, mname, 0, 'zero_shot', all_metrics(yte, p_te_src))
            for b in CFG['budgets']:
                buf = stratified_buffer(splits[tgt]['train'], b, seed)
                ybuf = buf['Label'].values
                # calibrate: frozen source model, Platt on buffer, source medians
                if not is_done(seed, src, tgt, mname, b, 'calibrate'):
                    Xbuf_src, _ = clean_X(buf, medians=src_med)
                    lr = platt_fit(src_model.predict_proba(Xbuf_src)[:, 1], ybuf)
                    mark(seed, src, tgt, mname, b, 'calibrate',
                         all_metrics(yte, platt_apply(lr, p_te_src)))
                    del Xbuf_src
                # buffer_only: fresh model on buffer, buffer medians
                if not is_done(seed, src, tgt, mname, b, 'buffer_only'):
                    Xbuf, mbuf = clean_X(buf)
                    Xte_b, _ = clean_X(te, medians=mbuf)
                    mark(seed, src, tgt, mname, b, 'buffer_only',
                         fit_eval(mname, seed, Xbuf, ybuf, Xte_b, yte))
                    del Xbuf, Xte_b
                    gc.collect()
                # augment: source train + buffer, medians on the union
                if not is_done(seed, src, tgt, mname, b, 'augment'):
                    both = pd.concat([splits[src]['train'], buf], ignore_index=True)
                    Xa, ma = clean_X(both)
                    ya = both['Label'].values
                    Xte_a, _ = clean_X(te, medians=ma)
                    mark(seed, src, tgt, mname, b, 'augment',
                         fit_eval(mname, seed, Xa, ya, Xte_a, yte))
                    del both, Xa, Xte_a
                    gc.collect()
                # weighted_augment (ref seed only): buffer duplicated to ~25% mass
                if seed == CFG['ref_seed'] and \
                   not is_done(seed, src, tgt, mname, b, 'weighted_augment'):
                    K = max(1, int(round(len(splits[src]['train']) / (3 * len(buf)))))
                    both = pd.concat([splits[src]['train']] + [buf] * K,
                                     ignore_index=True)
                    Xw, mw = clean_X(both)
                    yw = both['Label'].values
                    Xte_w, _ = clean_X(te, medians=mw)
                    m = fit_eval(mname, seed, Xw, yw, Xte_w, yte)
                    m['dup_K'] = K
                    mark(seed, src, tgt, mname, b, 'weighted_augment', m)
                    del both, Xw, Xte_w
                    gc.collect()
            # oracle (ref seed only, budget-independent): full target-train retrain
            if seed == CFG['ref_seed'] and \
               not is_done(seed, src, tgt, mname, 1, 'oracle'):
                Xo, mo = clean_X(splits[tgt]['train'])
                yo = splits[tgt]['train']['Label'].values
                Xte_o, _ = clean_X(te, medians=mo)
                mark(seed, src, tgt, mname, 1, 'oracle',
                     fit_eval(mname, seed, Xo, yo, Xte_o, yte))
                del Xo, Xte_o
                gc.collect()
            if src_model is not None:
                del src_model, p_te_src
                gc.collect()

print('rows recorded:', len(done))

In [ ]:
import pandas as pd, csv

p = '/content/drive/MyDrive/drift-conference/results/nfv2/nfv2_budget_strategies.csv'

# read raw, split header rows (appended blocks each wrote their own header) from data
rows = list(csv.reader(open(p)))
headers = [r for r in rows if r and r[0] == 'seed']
widest = max(headers, key=len)
print('header variants:', {len(h) for h in headers}, '| using', len(widest), 'cols')

recs = []
for r in rows:
    if not r or r[0] == 'seed':
        continue
    d = dict(zip(widest, r + [None] * (len(widest) - len(r))))
    recs.append(d)

df = pd.DataFrame(recs)
for c in df.columns:
    if c not in ('source', 'target', 'model', 'strategy'):
        df[c] = pd.to_numeric(df[c], errors='coerce')

print('rows recovered:', len(df))
print('seed counts:', df.seed.value_counts().to_dict())
print(df.groupby('strategy')['mcc'].agg(['count', 'mean', 'min', 'max']).round(3))

# drop the partial seed, rewrite clean with one consistent header
df = df[df.seed != 44]
df.to_csv(p, index=False)
print('\nrewritten clean:', len(df), 'rows |', df.seed.value_counts().to_dict())

In [ ]:
import pandas as pd
p = '/content/drive/MyDrive/drift-conference/results/nfv2/nfv2_budget_strategies.csv'
df = pd.read_csv(p)
c = df[df.strategy == 'calibrate']
print(c[['auprc_macro', 'fp_rate', 'brier', 'ece', 'macro_f1']].describe().round(4))
print('\nzero_shot AUPRC for comparison:')
print(df[df.strategy == 'zero_shot'][['auprc_macro', 'mcc']].describe().round(4))

In [ ]:
df = pd.read_csv(BUDGET_CSV).drop_duplicates(
        ['seed', 'source', 'target', 'model', 'budget', 'strategy'])
core = df[df.strategy.isin(['zero_shot', 'calibrate', 'buffer_only', 'augment'])]
agg = (core.groupby(['source', 'target', 'model', 'budget', 'strategy'])
           [['mcc', 'macro_f1', 'ece', 'auprc_macro', 'fp_rate']]
           .agg(['mean', 'std']).round(4))
agg.columns = ['_'.join(c) for c in agg.columns]
agg = agg.reset_index()
agg.to_csv(f'{RESULT}/nfv2_budget_aggregate.csv', index=False)

print('MCC by strategy and budget (mean over seeds):')
for src, tgt in DIRECTIONS:
    print(f'\n--- {src} -> {tgt} ---')
    sub = df[(df.source == src) & (df.target == tgt)]
    display(sub.pivot_table(index=['model', 'strategy'], columns='budget',
                            values='mcc', aggfunc='mean').round(3))

In [ ]:
import subprocess, glob, shutil
os.chdir(BASE)
cred = glob.glob('/content/drive/MyDrive/*/.git-credentials')
assert cred, 'no .git-credentials dotfile found in Drive'
shutil.copy(cred[0], '/root/.git-credentials')
!git config --global credential.helper store
!git config --global user.name "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"

nb = 'notebooks/08_equal_label_budget.ipynb'
!jupyter nbconvert --ClearOutputPreprocessor.enabled=True --inplace {nb}
targets = [nb, 'results/nfv2/nfv2_budget_strategies.csv',
           'results/nfv2/nfv2_budget_aggregate.csv']
for t in targets:
    assert os.path.exists(t), f'missing {t}'
    !git add {t}
staged = subprocess.run(['git', 'diff', '--cached', '--name-only'],
                        capture_output=True, text=True).stdout.split()
assert set(staged) <= set(targets), f'unexpected staged: {set(staged) - set(targets)}'
missing = [t for t in targets if t not in staged]
assert not missing, f'NOT staged: {missing}'
!git commit -m "08: equal-label-budget calibrate-vs-adapt (1/5/10% buffers, 5 strategies, 3 models, both directions)"
!git push
!git log --oneline -1
!git ls-files results/nfv2/